In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = sns.load_dataset("diamonds")

### Cleaning

In [4]:
print(df.isnull().sum())

carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64


In [5]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

146
(53794, 10)


In [6]:
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]
print(df.shape)

(53775, 10)


In [7]:
df = df[(df["y"] < 20) & (df["z"] < 10)]
print(df.shape)

(53772, 10)


In [8]:
print(df.describe())

              carat         depth         table         price             x  \
count  53772.000000  53772.000000  53772.000000  53772.000000  53772.000000   
mean       0.797525     61.748283     57.457911   3931.137321      5.731557   
std        0.473150      1.429626      2.233262   3985.853004      1.118590   
min        0.200000     43.000000     43.000000    326.000000      3.730000   
25%        0.400000     61.000000     56.000000    951.000000      4.710000   
50%        0.700000     61.800000     57.000000   2401.000000      5.700000   
75%        1.040000     62.500000     59.000000   5324.000000      6.540000   
max        5.010000     79.000000     95.000000  18823.000000     10.740000   

                  y             z  
count  53772.000000  53772.000000  
mean       5.733446      3.539325  
std        1.110500      0.691068  
min        3.680000      1.070000  
25%        4.720000      2.910000  
50%        5.710000      3.530000  
75%        6.540000      4.030000  


In [9]:
print(df.isnull().sum().sum())
print(df.duplicated().sum())
print((df[["x", "y", "z"]] <= 0).sum())
print(df[["x", "y", "z"]].max())

0
0
x    0
y    0
z    0
dtype: int64
x    10.74
y    10.54
z     6.98
dtype: float64


In [10]:
df = df.reset_index(drop=True)
df.shape

(53772, 10)

### Splitting

In [11]:
y = df["price"]
X = df.drop(columns=["price"])
print(X.shape)
print(y.shape)

(53772, 9)
(53772,)


In [12]:
ordinal_cols = ["cut", "color", "clarity"]
numeric_cols = ["carat", "depth", "table", "x", "y", "z"]
print(ordinal_cols)
print(numeric_cols)

['cut', 'color', 'clarity']
['carat', 'depth', 'table', 'x', 'y', 'z']


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(43017, 9)
(10755, 9)
(43017,)
(10755,)


### preprocessing

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [17]:
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

In [ ]:
preprocessor = ColumnTransformer([
    ("scaler", StandardScaler(), numeric_cols),
    ("ord", OrdinalEncoder(categories=[cut_order, color_order, clarity_order]), ordinal_cols)
])

In [20]:
linear_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

### Training & Cross validation

In [23]:
linear_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['carat','cut','color',...,'x','y','z']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of

In [24]:
print(linear_pipe.score(X_train, y_train))
print(rf_pipe.score(X_train, y_train))

0.908894146189991
0.9973870928059442


In [25]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [26]:
linear_scores = cross_val_score(
    linear_pipe, X_train, y_train,
    cv=kf, scoring="neg_root_mean_squared_error"
)

rf_scores = cross_val_score(
    rf_pipe, X_train, y_train,
    cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1
)

In [27]:
linear_rmse = -linear_scores
rf_rmse = -rf_scores

print("Linear Regression RMSE per fold:", linear_rmse.round(2))
print("Mean:", linear_rmse.mean().round(2))
print("Std:", linear_rmse.std().round(2))

print("Random Forest RMSE per fold:", rf_rmse.round(2))
print("Mean:", rf_rmse.mean().round(2))
print("Std:", rf_rmse.std().round(2))

Linear Regression RMSE per fold: [1232.23 1162.83 1221.   1215.71 1206.05]
Mean: 1207.56
Std: 23.91
Random Forest RMSE per fold: [556.55 571.58 559.4  533.68 536.34]
Mean: 551.51
Std: 14.41
